# Weekly Project 5 
## Implementation of global registration 
### Task 1
Today, your task is to implement a global registration algorithm.

It should be able to roughly align two point clouds.
Implement the global registration, and then try the following:

1. Can you fit `r1.pcd` and `r2.pcd`?
2. Can you fit `car1.ply` and `car2.ply`?
The corresponding files are in the `global_registration` folder.

Following code is all just copy paste from the exercise on Monday.

In [83]:
import open3d as o3d
import numpy as np
import copy

# helper function for drawing
# If you want it to be more clear set recolor=True
def draw_registrations(source, target, transformation = None, recolor = False):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    if(recolor):
        source_temp.paint_uniform_color([1, 0.706, 0])
        target_temp.paint_uniform_color([0, 0.651, 0.929])
    if(transformation is not None):
        source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp])

We have been running the same code found below just changing the file name in the reading line. The cars align very easily. The office room aligns a little worse with fit around .5 instead of 0.7 for the cars.

In [84]:
source = o3d.io.read_point_cloud("global_registration/car1.ply")
target = o3d.io.read_point_cloud("global_registration/car2.ply")

# Used for downsampling.
voxel_size = 0.05

# Show models side by side
#draw_registrations(source, target)

The above voxel_size is a good middle ground. Going to 0.1 gives unacceptable alignment, and going to 0.01 makes runtime too high (over 2 minutes compared to 10 seconds at 0.05).

In [85]:
####
# Downsample and find features here
####
source_d = source.voxel_down_sample(voxel_size=voxel_size)
target_d = target.voxel_down_sample(voxel_size=voxel_size)

radius_normal = voxel_size * 2
radius_feature = voxel_size * 5

source_d.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(
        radius=radius_normal,
        max_nn=30
    )
)

source_features = o3d.pipelines.registration.compute_fpfh_feature(
    source_d,
    o3d.geometry.KDTreeSearchParamHybrid(
        radius=radius_feature,
        max_nn=100
    )
)

target_d.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(
        radius=radius_normal,
        max_nn=30
    )
)

target_features = o3d.pipelines.registration.compute_fpfh_feature(
    target_d,
    o3d.geometry.KDTreeSearchParamHybrid(
        radius=radius_feature,
        max_nn=100
    )
)


print(source_features)

Feature class with dimension = 33 and num = 3490
Access its data via data member.


In [86]:
####
# Call RANSAC here
####

point_to_point =  o3d.pipelines.registration.TransformationEstimationPointToPoint(False)


ransac_result_ptp = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_d, target_d, 
    source_features, target_features, 
    max_correspondence_distance = voxel_size * 1.5,
    estimation_method=point_to_point, mutual_filter=True,
    checkers=[
    o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
    o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(voxel_size * 1.5),
    ],
    criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(100000, 0.9999))

# Parameters
threshold = 0.02

#Evaluate registration
print("Ransac alignment")
evaluation = o3d.pipelines.registration.evaluate_registration(source, target, threshold, ransac_result_ptp.transformation)
print("Fitness:", evaluation.fitness)
print("RMSE:", evaluation.inlier_rmse)
#draw_registrations(source, target, ransac_result_ptp.transformation, True)

Ransac alignment
Fitness: 0.9520509210795461
RMSE: 0.008317759395709745


I notice in the print above that the fitness varies ALOT. With current parameters it goes between 0.4 and 0.6 when using the office room. It shows ransac really is random.


### Task 2 (Challange)
Challanges attempt either or both:
- Implement local registration.

- Attempt to reconstruct the car from the images in `car_challange` folder.

You can use the notebooks from Monday as a starting point.

In [87]:
#Code is copy paste from notebook on monday

point_to_point =  o3d.pipelines.registration.TransformationEstimationPointToPoint()

target.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(
        radius=5,
        max_nn=30
    )
)
source.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(
        radius=5,
        max_nn=30
    )
)


# ransac_result_ptp.transformation is the result from task 1, use that as initial transform
icp_result = o3d.pipelines.registration.registration_icp(
    source, target, threshold,  ransac_result_ptp.transformation,
    point_to_point)
    
#Evaluate registration
print("Final alignment using ICP")
evaluation = o3d.pipelines.registration.evaluate_registration(source, target, threshold, icp_result.transformation)
print("Fitness:", evaluation.fitness)
print("RMSE:", evaluation.inlier_rmse)
#draw_registrations(source, target, icp_result.transformation, True)

Final alignment using ICP
Fitness: 0.98633721979569
RMSE: 0.005407374330258069
